# Assessment Notebook

## Imports
All modules used in this notebook are imported once here, following the [PEP 8](https://peps.python.org/pep-0008/#imports) convention: standard library modules first, then third-party modules.

In [1]:
# Third-party
import numpy as np

## Problem 0: GitHub Issues

I created a GitHub Issue for each problem in this assessment (Problems 0 to 6), so every piece of work has its own place to track progress.

### Why they are useful
- They break the assessment into small, clear tasks instead of one big piece of work.
- Each issue holds a checklist of what the problem requires, so I can see what is done and what is left.
- They leave a visible record of my progress for anyone reviewing the repository.

### How I will use them
- Comment on the relevant issue as I work, noting what I finished or what I found difficult.
- Reference the issue in my commit messages where it helps link the work to the task.
- Close each issue once I consider its problem complete.

## Problem 1: Representing SHA-256 Data

This problem covers how each type of data used by SHA-256 is represented in Python:

1. 32-bit words
2. Sequences of 32-bit words
3. The input message
4. 512-bit message blocks
5. The final 256-bit hash value
6. Big-endian integers

### Sub-section 1: 32-bit words

A **32-bit word** is a number made up of exactly 32 bits, so it can hold any
value from `0` up to `4,294,967,295` (2<sup>32</sup> − 1). SHA-256 uses these 32-bit words
throughout its calculations. If a calculation would go above this maximum,
the result wraps back around to `0` instead of growing bigger.

#### Research
To refresh my understanding of hashing before starting this assessment, I
looked up some information on [GeeksforGeeks](https://www.geeksforgeeks.org/computer-networks/sha-256-and-sha-3/),
covering how hashing works in general and the different types of hashing
algorithms that exist.

For this specific problem, I also looked into how Python represents numbers.
Python's built-in `int` type has no fixed size it just keeps growing as
needed, with no limit. This doesn't match what SHA-256 requires, since the
standard expects every value to stay within 32 bits and wrap around when it
goes over. NumPy's `uint32` type behaves this way, making it a good choice
for representing 32-bit words in this implementation.

In [2]:
# A 32-bit word can hold values from 0 to 2**32 - 1
word = np.uint32(4294967295)
print(word)

4294967295


In [3]:

#Demonstrating that `numpy.uint32` wraps around on overflow, rather than
#growing like a native Python `int`.

# NumPy uint32 wraps around at 2**32
wrapped = np.uint32(4294967295) + np.uint32(1)
print(wrapped)  # expect 0

# Compare to plain Python int, which just keeps growing
plain = 4294967295 + 1
print(plain)  # expect 4294967296

0
4294967296


C:\Users\Bogdan Nwachukwu\AppData\Local\Temp\ipykernel_31724\4034841768.py:5: RuntimeWarning: overflow encountered in scalar add
  wrapped = np.uint32(4294967295) + np.uint32(1)


### Sub-section 2: Sequences of 32-bit words

A **sequence of 32-bit words** is simply a list of several 32-bit numbers,
kept together in order. SHA-256 needs this for things like the eight
starting hash values and the 64 values used in each round of processing.

#### Research
A plain Python `list` can hold multiple numbers, but it doesn't enforce
that every value stays within 32 bits it would happily hold a mix of
small numbers, huge numbers, or even non-numbers. NumPy arrays, however,
can be given a fixed `dtype` (data type), so every single value in the
array is forced to be a proper 32-bit unsigned integer, matching what
SHA-256 requires throughout.

In [4]:
# A sequence of 32-bit words, stored as a NumPy array
words = np.array([1, 2, 3, 4294967295], dtype=np.uint32)
print(words)
print(words.dtype)

[         1          2          3 4294967295]
uint32


In [5]:
#Testing. Adding 1 to the last word should wrap it around to 0
words[-1] = words[-1] + np.uint32(1)
print(words)

[1 2 3 0]


C:\Users\Bogdan Nwachukwu\AppData\Local\Temp\ipykernel_31724\1565870136.py:2: RuntimeWarning: overflow encountered in scalar add
  words[-1] = words[-1] + np.uint32(1)


### Sub-section 3: Input message

The **input message** is the original data someone wants to hash for
example, a piece of text. SHA-256 doesn't work on text directly; it
works on raw bytes, so the message needs to be represented as a
sequence of bytes before any hashing can begin.

#### Research
In Python, text is normally stored as a `str`, but `str` represents
characters, not raw binary data, and can't be directly processed as
bits and bytes the way SHA-256 requires. Python's built-in `bytes`
type is the natural fit instead it's an immutable sequence of raw
byte values (`0`–`255`), exactly matching how the Secure Hash Standard
describes a message. Any text first needs to be *encoded* (e.g. using
UTF-8) to convert it into this `bytes` form.

In [6]:
# Representing an input message as bytes
message = "hello".encode("utf-8")
print(message)
print(type(message))

b'hello'
<class 'bytes'>


In [7]:
# Test with a simple ASCII message
msg1 = "hello".encode("utf-8")
print(msg1)

# Test with an empty message
msg2 = "".encode("utf-8")
print(msg2)

# Test with a non-ASCII character
msg3 = "héllo".encode("utf-8")
print(msg3)

b'hello'
b''
b'h\xc3\xa9llo'


### Sub-section 4: 512-bit message blocks

A **512-bit message block** is a fixed-size chunk of 64 bytes (512 bits).
SHA-256 processes the input message in these fixed-size chunks, one at
a time, rather than all at once. Each block is further split into
sixteen 32-bit words for processing.

#### Research
Since a 32-bit word is represented using `numpy.uint32`, and a sequence
of words is represented using a NumPy array, it makes sense to
represent a 512-bit block the same way: as a NumPy array of sixteen
`uint32` values (16 words × 32 bits = 512 bits). This keeps the
representation consistent with the earlier choices, and matches how
the Secure Hash Standard describes each block being split into words
for processing.

In [8]:
# A 512-bit block represented as sixteen 32-bit words
block = np.zeros(16, dtype=np.uint32)
print(block)
print(len(block), "words")
print(block.nbytes, "bytes")

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
16 words
64 bytes


In [9]:
# Testing. Confirm total size matches 512 bits (64 bytes)
assert len(block) == 16
assert block.nbytes == 64
print("Block size check passed: 16 words, 64 bytes, 512 bits")

Block size check passed: 16 words, 64 bytes, 512 bits


### Sub-section 5: Final 256-bit hash value

The **final 256-bit hash value** is the end result of SHA-256. It is made
up of eight 32-bit words joined together (8 × 32 = 256 bits), produced
after processing every block of the message. It is usually shown to
people as a 64-character hexadecimal string, rather than raw bits.

In [10]:
# Example final hash state: eight 32-bit words
hash_words = np.array([0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
                        0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19],
                       dtype=np.uint32)

# Convert to a 64-character lowercase hex string
hash_hex = "".join(f"{word:08x}" for word in hash_words)
print(hash_hex)
print(len(hash_hex), "characters")

6a09e667bb67ae853c6ef372a54ff53a510e527f9b05688c1f83d9ab5be0cd19
64 characters


In [11]:
#Testing that the hash is indeed 64 hex characters (256 bits)
assert len(hash_hex) == 64
print("Hash length check passed: 64 hex characters (256 bits)")

Hash length check passed: 64 hex characters (256 bits)


### Sub-section 6: Big-endian integers

**Big-endian** is a rule for the order bytes are stored or read in when
representing a number. In big-endian, the most significant byte (the
biggest, leftmost part of the number) comes first. SHA-256 requires
big-endian ordering when converting between bytes and integers.

In [12]:
data = bytes([0x01, 0x02, 0x03, 0x04])

big_endian_value = int.from_bytes(data, byteorder="big")
little_endian_value = int.from_bytes(data, byteorder="little")

print("Bytes:", data)
print("Big-endian value:", hex(big_endian_value))
print("Little-endian value:", hex(little_endian_value))

Bytes: b'\x01\x02\x03\x04'
Big-endian value: 0x1020304
Little-endian value: 0x4030201


In [13]:

#Testing

original = 0x01020304
as_bytes = original.to_bytes(4, byteorder="big")
back_to_int = int.from_bytes(as_bytes, byteorder="big")

assert back_to_int == original
print("Round-trip check passed:", hex(back_to_int))

Round-trip check passed: 0x1020304


## Problem 2: SHA-256 Bitwise Operations

| Function | Standard notation | Definition |
|---|---|---|
| `Ch(x, y, z)` | Ch(x, y, z) | (x ∧ y) ⊕ (¬x ∧ z) |
| `Maj(x, y, z)` | Maj(x, y, z) | (x ∧ y) ⊕ (x ∧ z) ⊕ (y ∧ z) |
| `Sigma0(x)` | Σ<sub>0</sub><sup>{256}</sup>(x) | ROTR<sup>2</sup>(x) ⊕ ROTR<sup>13</sup>(x) ⊕ ROTR<sup>22</sup>(x) |
| `Sigma1(x)` | Σ<sub>1</sub><sup>{256}</sup>(x) | ROTR<sup>6</sup>(x) ⊕ ROTR<sup>11</sup>(x) ⊕ ROTR<sup>25</sup>(x) |
| `sigma0(x)` | σ<sub>0</sub><sup>{256}</sup>(x) | ROTR<sup>7</sup>(x) ⊕ ROTR<sup>18</sup>(x) ⊕ SHR<sup>3</sup>(x) |
| `sigma1(x)` | σ<sub>1</sub><sup>{256}</sup>(x) | ROTR<sup>17</sup>(x) ⊕ ROTR<sup>19</sup>(x) ⊕ SHR<sup>10</sup>(x) |

Research
imports at the top
more comments
add references https://nvlpubs.nist.gov/nistpubs/fips/nist.fips.180-4.pdf
add youtube video https://youtu.be/orIgy2MjqrA?si=Cwixfq9k3-PkO6_4


### Sub-section 1: Ch Function

In [14]:
WORD_BITS = 32

def _rotr(value, amount):
    value = np.uint32(value)
    return np.uint32((value >> amount) | (value << (WORD_BITS - amount)))

def _shr(value, amount):
    return np.uint32(np.uint32(value) >> amount)

In [15]:
def Ch(sel, a, b):
    sel = np.uint32(sel)
    a = np.uint32(a)
    b = np.uint32(b)
    return np.uint32(b ^ (sel & (a ^ b)))

In [16]:
def test_ch():
    assert Ch(0xFFFFFFFF, 0x9ABCDEF0, 0x0F1E2D3C) == 0x9ABCDEF0
    assert Ch(0x00000000, 0x9ABCDEF0, 0x0F1E2D3C) == 0x0F1E2D3C
    assert Ch(0b0110, 0b1011, 0b1101) == 0b1011
    print("All Ch tests passed")

test_ch()

All Ch tests passed


### Sub-section 2: Maj Function

In [17]:
def Maj(a, b, c):
    a = np.uint32(a)
    b = np.uint32(b)
    c = np.uint32(c)
    return np.uint32((a & b) | (c & (a | b)))

In [18]:
def test_Maj():
    assert Maj(0xA5A5A5A5, 0xA5A5A5A5, 0xA5A5A5A5) == 0xA5A5A5A5
    assert Maj(0xFFFFFFFF, 0xFFFFFFFF, 0x00000000) == 0xFFFFFFFF
    assert Maj(0b1001, 0b1100, 0b0101) == 0b1101
    print("All Maj tests passed")

test_Maj()

All Maj tests passed


### Sub-section 3: Sigma0 Function

In [19]:
def Sigma0(word):
    SIGMA0_ROTATIONS = (2, 13, 22)
    
    r1, r2, r3 = SIGMA0_ROTATIONS
    return np.uint32(_rotr(word, r1) ^ _rotr(word, r2) ^ _rotr(word, r3))

In [20]:
def test_Sigma0():
    assert Sigma0(0xAAAAAAAA) == 0x55555555
    assert Sigma0(0x00000100) == 0x08040040
    assert Sigma0(0x12345678) == 0x66146474
    print("All Sigma0 tests passed")

test_Sigma0()

All Sigma0 tests passed


### Sub-section 4: Sigma1 Function 

In [21]:
SIGMA1_ROTATIONS = (6, 11, 25)


def Sigma1(word):
    first, second, third = SIGMA1_ROTATIONS
    mixed = _rotr(word, first) ^ _rotr(word, second)
    return np.uint32(mixed ^ _rotr(word, third))

In [22]:
def test_Sigma1():
    assert Sigma1(0x55555555) == 0x55555555
    assert Sigma1(0x00008000) == 0x00400210
    assert Sigma1(0xDEADBEEF) == 0x345E14A3
    print("All Sigma1 tests passed")


test_Sigma1()

All Sigma1 tests passed


### Sub-section 5: sigma0 Function

In [23]:
SMALL_SIGMA0_ROTATIONS = (7, 18)
SMALL_SIGMA0_SHIFT = 3


def sigma0(word):
    first, second = SMALL_SIGMA0_ROTATIONS
    mixed = _rotr(word, first) ^ _rotr(word, second)
    return np.uint32(mixed ^ _shr(word, SMALL_SIGMA0_SHIFT))

In [24]:
def test_sigma0():
    assert sigma0(0x00000008) == 0x10020001
    assert sigma0(0xFFFFFFFF) == 0x1FFFFFFF
    assert sigma0(0xCAFEBABE) == 0xCA65989D
    print("All sigma0 tests passed")


test_sigma0()

All sigma0 tests passed


### Sub-section 6: sigma1 Function

In [25]:
SMALL_SIGMA1_ROTATIONS = (17, 19)
SMALL_SIGMA1_SHIFT = 10


def sigma1(word):
    first, second = SMALL_SIGMA1_ROTATIONS
    mixed = _rotr(word, first) ^ _rotr(word, second)
    return np.uint32(mixed ^ _shr(word, SMALL_SIGMA1_SHIFT))

In [26]:
def test_sigma1():
    assert sigma1(0x00000400) == 0x02800001
    assert sigma1(0xFFFFFFFF) == 0x003FFFFF
    assert sigma1(0x0BADF00D) == 0x4605CFDF
    print("All sigma1 tests passed")


test_sigma1()

All sigma1 tests passed


---

*End of Notebook*